<a href="https://colab.research.google.com/github/iav2002/Assignment_Advanced_Topics_In_DeepLearning/blob/main/Part3_DQN_Variants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 3: DQN Variants on Warehouse Navigation

We will train three Deep Q-Network variants on a 5x5 warehouse grid environment, using the visual embeddings of sign, wall, floor pallet.

 The goal is to isolate the contribution of two well known DQN components: the experience replay buffer and the target network.

  We train them in an additive fashion (vanilla → +replay → +target) and compare learning dynamics across the three.

The reward will make sense on the second run of the notebook

In [1]:
import sys, os, json, random, time
from pathlib import Path
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdvancedDL')
EMB_PATH = DRIVE_ROOT / 'embeddings' / 'mean_embeddings.npy'
ENV_PATH = DRIVE_ROOT / 'WarehouseEnv.py'
RESULTS_ROOT = DRIVE_ROOT / 'results_part3'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# sanity checks
assert EMB_PATH.exists(), f"missing: {EMB_PATH}"
assert ENV_PATH.exists(), f"missing: {ENV_PATH}"
print("paths ok")

Mounted at /content/drive
paths ok


In [3]:
sys.path.insert(0, str(DRIVE_ROOT))
import gymnasium  # colab by defailt but in case
from WarehouseEnv import WarehouseEnv
print("env imported ok")

env imported ok


## Loading the Part 2 embeddings

The mean embeddings come from Part 2's best ResNet50 model (Variant C of Experiment 1, 99.05% test accuracy). They are stored as a dict keyed by class name (`'floor'`, `'wall'`, `'pallet'`, `'sign'`), each pointing to a 2048 dim vector.

The environment, however, looks them up by integer ID (0=floor, 1=wall, 2=pallet, 3=sign), so we remap before passing to the env.


In [4]:
emb_str = np.load(EMB_PATH, allow_pickle=True).item()
print("classes:", list(emb_str.keys()))
print("vector dim:", emb_str['floor'].shape)

# remap str keys to int IDs the env expects
NAME_TO_ID = {'floor': 0, 'wall': 1, 'pallet': 2, 'sign': 3}
emb = {NAME_TO_ID[k]: v.astype(np.float32) for k, v in emb_str.items()}
print("remapped keys:", list(emb.keys()))

classes: ['floor', 'wall', 'pallet', 'sign']
vector dim: (2048,)
remapped keys: [0, 1, 2, 3]


## Grid layout

We start with a minimal but non-trivial 5x5 grid: the agent spawns at (0,0), the pallet sits at (4,4), and a single wall sits at the center to force the agent to learn a detour. No signs in this first version. Once the three DQN variants are working we'll revisit this layout to make better use of the visual embeddings.

Coordinate convention: `grid_map[row, col]`. The agent's position is stored as `(x, y)` where x is column and y is row, matching the env's movement code.

Will add complexity once it works

In [5]:
# 0=floor, 1=wall, 2=pallet, 3=sign
GRID = [
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 2],
]

# quick visual check
sym = {0:'.', 1:'#', 2:'P', 3:'S'}
for row in GRID:
    print(' '.join(sym[c] for c in row))

. . . . .
. . . . .
. . # . .
. . . . .
. . . . P
